# 06 SVM Tuning Experiments

SVM-focused TF-IDF hyperparameter tuning and class imbalance experiments for the final 1,000-row MBG labeled sample. Macro F1 is prioritized because the dataset is imbalanced, especially for the `negatif` class. These are tuning experiments, not final thesis conclusions.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "mbg_labeled_sample_1000.csv"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

## Load Final Labeled Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head()

## Label Distribution

In [ ]:
label_counts = df["label"].value_counts().rename("count")
label_distribution = label_counts.to_frame()
label_distribution["percentage"] = (label_distribution["count"] / len(df) * 100).round(2)
label_distribution

## Run SVM Tuning Script Logic

In [ ]:
from run_svm_tuning_experiments import run_svm_tuning_experiments

summary = run_svm_tuning_experiments()
summary["best_svm"]

## Test Results

In [ ]:
test_results = pd.read_csv(REPORTS_DIR / "15_svm_tuning_test_results.csv")
test_results

## Best Parameters

In [ ]:
with open(REPORTS_DIR / "15_svm_tuning_summary.json", encoding="utf-8") as file_obj:
    saved_summary = json.load(file_obj)

best_params = pd.DataFrame(
    [
        {
            "experiment": key,
            "display_name": value["display_name"],
            "best_cv_macro_f1": value["best_cv_macro_f1"],
            "best_params": json.dumps(value["best_params"], ensure_ascii=False, sort_keys=True),
        }
        for key, value in saved_summary["experiments"].items()
    ]
)
best_params

## Confusion Matrices

In [ ]:
confusion_matrix_paths = [
    FIGURES_DIR / "15_confusion_matrix_tuned_linearsvc.png",
    FIGURES_DIR / "15_confusion_matrix_tuned_svc_linear.png",
    FIGURES_DIR / "15_confusion_matrix_tuned_svc_rbf.png",
]
for path in confusion_matrix_paths:
    print(path.name)
    display(Image(filename=str(path)))

## Best SVM by Macro F1

In [ ]:
best_svm = saved_summary["best_svm"]
print(f"Best tuned SVM by test macro F1: {best_svm['experiment_display_name']} ({best_svm['experiment']})")
print(f"Test macro F1: {best_svm['test_f1_macro']:.4f}")
print(f"Model artifact: {best_svm['artifact_path']}")

comparison = saved_summary["comparison_to_logistic_regression_baseline"]
if comparison["baseline_available"]:
    print(f"Logistic Regression baseline macro F1: {comparison['logistic_regression_f1_macro']:.4f}")
    print(f"Tuned SVM underperforms Logistic Regression baseline: {comparison['tuned_svm_underperforms']}")